# Notebook OOP-2 — House rules — hiding details, families & backpacks

| | |
|---|---|
| **Focus** | Object-oriented Python (Intermediate) |
| **Daily-life theme** | piggy banks, bikes vs cars, backpacks packing lists |
| **Pattern** | Story → Concept → Demo → **Exercise** → Solution |

**Prerequisites:** `01-oop-beginner-blueprints-and-objects.ipynb`

**Next notebook:** `03-oop-advanced-contracts-and-plugins.ipynb`

---

## Learning objectives

- Use **encapsulation** (leading `_`) like a PIN panel hiding balance digits.
- Tell **inheritance** (`Vehicle`) from **composition** (`Backpack` holds pens).
- Optional `@property` for validation — thermostat can't read −400°F.

## Table of contents

1. **Encapsulation — piggy bank**
2. **`@property` thermostat bounds**
3. **Inheritance — vehicles**
4. **Composition — backpack**
5. ****Progressive drills** — mailbox → fuel tank → coach roster**
6. **Exercise — gym membership tiers**

---

> **Tip:** When an analogy clicks, rename classes to AI-flavored names (`Embedder`, `ToolRegistry`) — same mechanics.

---


## 1 · Encapsulation — piggy bank hides counting coins

**Daily life:** Kids shake a sealed piggy bank—they hear coins but can't silently steal without **breaking rules**. Methods (`deposit`, `safe_balance_hint`) are the **official doors**.

Python convention: `_balance` signals "please don't touch directly"; technically possible, but rude like popping open someone's bank.


In [ ]:
class PiggyBank:
    def __init__(self) -> None:
        self._cents = 0

    def deposit(self, dollars: float) -> None:
        if dollars < 0:
            raise ValueError("no stealing via negative deposits")
        self._cents += int(round(dollars * 100))

    def shake_estimate(self) -> str:
        if self._cents == 0:
            return "empty"
        if self._cents < 500:
            return "light"
        return "heavy"


pig = PiggyBank()
pig.deposit(3.40)
pig.deposit(2)
print(pig.shake_estimate())


## 2 · `@property` — thermostat clamp

**Daily life:** Wall thermostat displays °F but refuses absurd targets—same idea as getters/setters with validation.


In [ ]:
class Thermostat:
    def __init__(self) -> None:
        self._temp_f = 68.0

    @property
    def temp_f(self) -> float:
        return self._temp_f

    @temp_f.setter
    def temp_f(self, value: float) -> None:
        if not (55 <= value <= 85):
            raise ValueError("unrealistic comfort zone")
        self._temp_f = value


t = Thermostat()
t.temp_f = 72
print(t.temp_f)


## 3 · Inheritance — bicycles & cars share "start journey"

**Daily life:** Different vehicles **share** ideas (they move people) but implement movement differently.


In [ ]:
class Vehicle:
    def __init__(self, brand: str) -> None:
        self.brand = brand

    def commute_noise(self) -> str:
        raise NotImplementedError("each vehicle sounds different")


class Bicycle(Vehicle):
    def commute_noise(self) -> str:
        return "tick tick tick"


class Bus(Vehicle):
    def commute_noise(self) -> str:
        return "whoosh-air-brakes"


for v in (Bicycle("Trek"), Bus("CityLine")):
    print(v.brand, "->", v.commute_noise())


## 4 · Composition — backpack *has* supplies

**Daily life:** Backpack isn't a "kind of pen"; it **contains** pens—**has-a**, not **is-a**.

This mirrors AI stacks: `Pipeline` **has** `Retriever` + `LLMClient`.


In [ ]:
class Pen:
    def __init__(self, ink_color: str) -> None:
        self.ink_color = ink_color


class Backpack:
    def __init__(self) -> None:
        self.items: list[Pen] = []

    def pack(self, pen: Pen) -> None:
        self.items.append(pen)

    def palette(self) -> list[str]:
        return [p.ink_color for p in self.items]


bag = Backpack()
bag.pack(Pen("blue"))
bag.pack(Pen("black"))
print(bag.palette())


---

## Progressive examples — **easy → harder**

Same ideas as above—now layer **privacy**, **validated fuel**, **teams built from people**.


### A · Easy — **mailbox** (letters hidden inside)

**Daily life:** Slot accepts envelopes; nosy neighbors only see **how full**, not love-letter contents.


In [ ]:
class Mailbox:
    def __init__(self) -> None:
        self._letters: list[str] = []

    def deposit(self, note: str) -> None:
        if not note.strip():
            raise ValueError("empty mail")
        self._letters.append(note.strip())

    def count(self) -> int:
        return len(self._letters)


box = Mailbox()
box.deposit("dentist reminder")
box.deposit("postcard")
print("waiting letters:", box.count())


### B · Medium — **fuel tank** (`@property` + clamp)

**Daily life:** Gauge can't display negative gallons or overflow the tank.


In [ ]:
class FuelTank:
    def __init__(self, max_gallons: float, initial: float) -> None:
        self._cap = max_gallons
        if not (0 <= initial <= max_gallons):
            raise ValueError("bad fill level")
        self._gallons = initial

    @property
    def gallons(self) -> float:
        return self._gallons

    @gallons.setter
    def gallons(self, value: float) -> None:
        if not (0 <= value <= self._cap):
            raise ValueError("fuel must fit tank")
        self._gallons = value


tank = FuelTank(max_gallons=15, initial=10)
tank.gallons = tank.gallons - 3  # explicit read/set avoids augmented-assign quirks on properties
print("after highway stretch", tank.gallons)
tank.gallons = 14
print("filled to", tank.gallons)


### C · Harder — **coach roster** (composition over inheritance)

**Daily life:** Coach **has** athletes on a clipboard—coach isn't "a type of athlete"; it's a role managing many.


In [ ]:
class Athlete:
    def __init__(self, name: str, sport: str) -> None:
        self.name = name
        self.sport = sport


class Coach:
    def __init__(self, name: str) -> None:
        self.name = name
        self._roster: list[Athlete] = []

    def sign(self, athlete: Athlete) -> None:
        self._roster.append(athlete)

    def roster_summary(self) -> str:
        parts = [f"{a.name} ({a.sport})" for a in self._roster]
        return ", ".join(sorted(parts))


c = Coach("Rivera")
c.sign(Athlete("Bo", "soccer"))
c.sign(Athlete("Ali", "track"))
print(c.roster_summary())


### Exercise — gym membership tiers

**Daily life:** Basic vs Premium members **share** check-in but Premium unlocks sauna visits.

Requirements:

1. Base class `Member(name)` with method `benefits() -> list[str]` returning base perks (e.g. `["locker"]`).
2. `PremiumMember` adds `"sauna"` **without breaking** basic perks.
3. `describe(member)` prints `"Name — perk1, perk2"` sorted alphabetically.


In [ ]:
class Member:
    def __init__(self, name: str) -> None:
        raise NotImplementedError

    def benefits(self) -> list[str]:
        raise NotImplementedError


class PremiumMember(Member):
    def benefits(self) -> list[str]:
        raise NotImplementedError


def describe(member: Member) -> str:
    raise NotImplementedError


m = Member("Alex")
p = PremiumMember("Jordan")
assert "locker" in m.benefits()
assert "sauna" in p.benefits() and "locker" in p.benefits()
assert describe(p).startswith("Jordan —")
print("Exercise OK")


### Solution — gym tiers

<details>
<summary>Reveal solution</summary>

```python
class Member:
    def __init__(self, name: str) -> None:
        self.name = name

    def benefits(self) -> list[str]:
        return ["locker"]


class PremiumMember(Member):
    def benefits(self) -> list[str]:
        return sorted(super().benefits() + ["sauna"])


def describe(member: Member) -> str:
    perks = ", ".join(sorted(member.benefits()))
    return f"{member.name} — {perks}"
```

</details>
